In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# シンプルなAttentionを手で実装する
# 「犬が好きな私」の各単語を4次元のベクトルで表す（ダミー）
torch.manual_seed(42)

words = ["私", "が", "犬", "を", "好き"]
seq_len = len(words)
d_model = 4  # 各単語のベクトル次元数

# 各単語のベクトル（ランダムで初期化）
X = torch.randn(seq_len, d_model)
print("入力の形:", X.shape)
print(X.round(decimals=2))

入力の形: torch.Size([5, 4])
tensor([[ 1.9300,  1.4900,  0.9000, -2.1100],
        [-0.7600,  1.0800,  0.8000,  1.6800],
        [ 0.3600, -0.6900, -0.4900,  0.2400],
        [-0.2300,  0.0400, -0.2500,  0.8600],
        [-0.3100, -0.4000,  0.8000, -0.6200]])


In [ ]:
# Q・K・Vを計算するための重み行列（これも学習で更新される）
torch.manual_seed(0)
W_Q = torch.randn(d_model, d_model)
W_K = torch.randn(d_model, d_model)
W_V = torch.randn(d_model, d_model)

# Q・K・Vを計算する
Q = X @ W_Q  # (5, 4) @ (4, 4) → (5, 4) # 「何を探しているか」の視点
K = X @ W_K # 「自分はどんな情報か」の視点
V = X @ W_V # 「実際に渡す情報」

print("Qの形:", Q.shape)
print("Kの形:", K.shape)
print("Vの形:", V.shape)

Qの形: torch.Size([5, 4])
Kの形: torch.Size([5, 4])
Vの形: torch.Size([5, 4])


In [6]:
# AttentionスコアをQ・Kの内積で計算する
d_k = d_model  # キーの次元数

# スコア行列を計算する（どの単語にどれだけ注目するか）
scores = Q @ K.T / (d_k ** 0.5)  # (5, 4) @ (4, 5) → (5, 5)
print("スコア行列の形:", scores.shape)
print("\nスコア行列:\n", scores.detach().round(decimals=2))

スコア行列の形: torch.Size([5, 5])

スコア行列:
 tensor([[ 17.4600,  -7.6400,  -1.4800,  -4.7900,   2.4500],
        [-25.2900,   8.0800,   2.6100,   5.6100,  -0.9600],
        [  5.2400,  -2.3900,  -0.0800,  -1.1100,  -0.2200],
        [-10.1400,   2.5300,   1.3300,   2.2100,  -0.5900],
        [ 12.0800,  -1.0600,  -2.0200,  -2.1100,   0.1300]])


In [7]:
attention_weights = F.softmax(scores, dim=-1)  # 行ごとに正規化
print("Attention重み:\n", attention_weights.detach().round(decimals=2))

Attention重み:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.9200, 0.0000, 0.0800, 0.0000],
        [0.9900, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4800, 0.1500, 0.3500, 0.0200],
        [1.0000, 0.0000, 0.0000, 0.0000, 0.0000]])


In [8]:
# Attention重みをValueに掛けて出力を計算する
output = attention_weights @ V  # (5, 5) @ (5, 4) → (5, 4)
print("出力の形:", output.shape)
print("\n出力:\n", output.detach().round(decimals=3))

出力の形: torch.Size([5, 4])

出力:
 tensor([[-1.7210, -3.9060,  2.8360,  4.5230],
        [ 1.6420,  4.3630, -1.5870, -2.2590],
        [-1.7030, -3.8590,  2.7990,  4.4680],
        [ 1.0040,  2.8430, -1.2660, -1.7000],
        [-1.7210, -3.9050,  2.8360,  4.5230]])
